# Data Preprocessing and Validation Notebook

This notebook performs data preprocessing and validation on the customer communication dataset.
It profiles the data, detects missing values, applies imputation strategies, validates the results, and exports a cleaned dataset.

## Section 1: Data Loading & Profiling

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

try:
    import missingno as msno
    HAS_MISSINGNO = True
except ImportError:
    HAS_MISSINGNO = False
    print('missingno not installed, skipping matrix visualization')

In [ ]:
# Load the dataset
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv')

print(f'Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nRows: {df.shape[0]}')
print(f'Columns: {df.shape[1]}')

In [ ]:
# Data types of each column
print('=== Column Data Types ===')
print(df.dtypes)
print(f'\nTotal columns: {len(df.columns)}')

In [ ]:
# Preview first 5 rows
print('=== First 5 Rows ===')
df.head()

In [ ]:
# Descriptive statistics for numerical columns
print('=== Descriptive Statistics (Numerical) ===')
df.describe()

In [ ]:
# Unique value counts for categorical columns
print('=== Categorical Column Value Counts ===')
categorical_cols = df.select_dtypes(include=['object']).columns
for col in categorical_cols:
    print(f'\n--- {col} ({df[col].nunique()} unique values) ---')
    print(df[col].value_counts().head(10))

## Section 2: Missing Value Detection

In [ ]:
# Compute missing value count and percentage per column
missing_count = df.isnull().sum()
missing_percentage = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'column': df.columns,
    'missing_count': missing_count.values,
    'missing_percentage': missing_percentage.values.round(2)
})

print('=== Missing Value Summary ===')
print(f'Total missing values: {missing_count.sum()}')
print(f'Columns with missing values: {(missing_count > 0).sum()} / {len(df.columns)}')
print()
missing_summary

In [ ]:
# Visualize missing values
cols_with_missing = missing_summary[missing_summary['missing_count'] > 0].sort_values('missing_count', ascending=False)

if len(cols_with_missing) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.barplot(data=cols_with_missing, x='missing_count', y='column', palette='Reds_r', ax=ax)
    ax.set_title('Missing Values per Column', fontsize=14)
    ax.set_xlabel('Missing Count')
    ax.set_ylabel('Column')
    for i, (count, pct) in enumerate(zip(cols_with_missing['missing_count'], cols_with_missing['missing_percentage'])):
        ax.text(count + 50, i, f'{count} ({pct}%)', va='center', fontsize=9)
    plt.tight_layout()
    plt.show()
else:
    print('No missing values found in any column.')

# Missingno matrix if available
if HAS_MISSINGNO:
    print('\n=== Missing Value Matrix ===')
    msno.matrix(df, figsize=(14, 6))
    plt.show()

## Section 3: Numerical Column Imputation

Fill missing numerical values with the **median** of each column (computed from non-missing values only).

In [ ]:
numerical_cols = [
    'item_price', 'connected_handling_time', 'age', 'total_spend',
    'items_purchased', 'average_rating', 'discount_applied', 'days_since_last_purchase'
]

print('=== Numerical Column Imputation (Median) ===')
for col in numerical_cols:
    if col in df.columns:
        missing_before = df[col].isnull().sum()
        if missing_before > 0:
            median_val = df[col].median()
            df[col] = df[col].fillna(median_val)
            print(f'  {col}: filled {missing_before} missing values with median = {median_val}')
        else:
            print(f'  {col}: no missing values')
    else:
        print(f'  {col}: column not found in dataset')

## Section 4: Categorical Column Imputation

Fill missing categorical values with the **mode** (most frequent value). If mode cannot be determined, use `"Unknown"`.

In [ ]:
categorical_impute_cols = [
    'customer_message', 'customer_city', 'product_category',
    'gender', 'city', 'membership_type', 'satisfaction_level'
]

print('=== Categorical Column Imputation (Mode) ===')
for col in categorical_impute_cols:
    if col in df.columns:
        missing_before = df[col].isnull().sum()
        if missing_before > 0:
            mode_vals = df[col].mode()
            if len(mode_vals) > 0:
                fill_value = mode_vals[0]
            else:
                fill_value = 'Unknown'
            df[col] = df[col].fillna(fill_value)
            print(f'  {col}: filled {missing_before} missing values with "{fill_value}"')
        else:
            print(f'  {col}: no missing values')
    else:
        print(f'  {col}: column not found in dataset')

## Section 5: DateTime Column Imputation

Parse `order_date_time` to datetime, then apply **forward-fill** followed by **backward-fill** to handle missing values.

In [ ]:
# Parse datetime columns
datetime_cols = ['order_date_time', 'issue_reported_at', 'issue_responded', 'survey_response_date']

for col in datetime_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)

# Impute order_date_time with forward-fill then backward-fill
print('=== DateTime Column Imputation (order_date_time) ===')
if 'order_date_time' in df.columns:
    missing_before = df['order_date_time'].isnull().sum()
    print(f'  Missing before imputation: {missing_before}')
    
    # Forward fill
    df['order_date_time'] = df['order_date_time'].ffill()
    missing_after_ffill = df['order_date_time'].isnull().sum()
    ffill_count = missing_before - missing_after_ffill
    
    # Backward fill for any remaining
    df['order_date_time'] = df['order_date_time'].bfill()
    missing_after_bfill = df['order_date_time'].isnull().sum()
    bfill_count = missing_after_ffill - missing_after_bfill
    
    print(f'  Forward-fill: {ffill_count} values filled')
    print(f'  Backward-fill: {bfill_count} values filled')
    print(f'  Remaining missing: {missing_after_bfill}')

## Section 6: Post-Imputation Validation

Verify that all missing values have been resolved after imputation.

In [ ]:
# Recompute missing values after imputation
post_missing = df.isnull().sum()
total_missing = post_missing.sum()

print('=== Post-Imputation Validation ===')
print(f'Total missing values remaining: {total_missing}')

if total_missing > 0:
    remaining = post_missing[post_missing > 0]
    print('\n⚠️  WARNING: Some columns still have missing values:')
    for col, count in remaining.items():
        print(f'  {col}: {count} missing')
else:
    print('\n✅ Dataset is clean — no missing values remain.')

## Section 7: Data Type Validation

Verify that columns have the correct data types after preprocessing.

In [ ]:
print('=== Data Type Validation ===')

# Check numerical columns
for col in numerical_cols:
    if col in df.columns:
        if not pd.api.types.is_numeric_dtype(df[col]):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                print(f'  {col}: converted to numeric')
            except Exception as e:
                print(f'  {col}: conversion failed — {e}')
        else:
            print(f'  {col}: ✓ numeric ({df[col].dtype})')

# Check datetime columns
for col in datetime_cols:
    if col in df.columns:
        if not pd.api.types.is_datetime64_any_dtype(df[col]):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce', dayfirst=True)
                print(f'  {col}: converted to datetime')
            except Exception as e:
                print(f'  {col}: conversion failed — {e}')
        else:
            print(f'  {col}: ✓ datetime ({df[col].dtype})')

## Section 8: NLP Text Preprocessing

Prepare the `customer_message` column for NLP model creation:
- Lowercase text
- Remove special characters, emojis, and extra whitespace
- Tokenize
- Remove stopwords
- Lemmatize tokens
- Create a `cleaned_message` column ready for NLP pipelines

In [ ]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

# Download required NLTK data
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

print('NLTK resources loaded successfully.')

In [ ]:
def preprocess_text(text):
    """Clean and preprocess text for NLP."""
    if pd.isna(text) or not isinstance(text, str):
        return ''
    
    # Lowercase
    text = text.lower()
    
    # Remove emojis and non-ASCII characters
    text = text.encode('ascii', 'ignore').decode('ascii')
    
    # Remove special characters, keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stopwords and lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 1]
    
    return ' '.join(tokens)


# Apply text preprocessing
print('=== NLP Text Preprocessing ===')
print(f'Processing {len(df)} customer messages...')

df['cleaned_message'] = df['customer_message'].apply(preprocess_text)

# Show before/after examples
sample_idx = df[df['customer_message'].str.len() > 20].head(5).index
print('\n--- Before vs After Cleaning ---')
for idx in sample_idx:
    print(f'\nOriginal : {df.loc[idx, "customer_message"][:100]}')
    print(f'Cleaned  : {df.loc[idx, "cleaned_message"][:100]}')

In [ ]:
# Update word count and message length based on cleaned text
df['cleaned_word_count'] = df['cleaned_message'].apply(lambda x: len(x.split()) if x else 0)
df['cleaned_message_length'] = df['cleaned_message'].str.len()

# Stats on cleaned messages
empty_count = (df['cleaned_message'] == '').sum()
print(f'\n=== Cleaned Message Stats ===')
print(f'Total messages: {len(df)}')
print(f'Empty after cleaning: {empty_count} ({empty_count/len(df)*100:.1f}%)')
print(f'Non-empty messages: {len(df) - empty_count}')
print(f'\nWord count distribution (non-empty):')
non_empty = df[df['cleaned_message'] != '']['cleaned_word_count']
print(non_empty.describe())

In [ ]:
# Visualize word count distribution after NLP preprocessing
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Word count distribution
non_empty_wc = df[df['cleaned_message'] != '']['cleaned_word_count']
axes[0].hist(non_empty_wc, bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Word Count Distribution (Cleaned Messages)', fontsize=12)
axes[0].set_xlabel('Word Count')
axes[0].set_ylabel('Frequency')

# Message length distribution
non_empty_ml = df[df['cleaned_message'] != '']['cleaned_message_length']
axes[1].hist(non_empty_ml, bins=30, color='coral', edgecolor='white')
axes[1].set_title('Message Length Distribution (Cleaned Messages)', fontsize=12)
axes[1].set_xlabel('Character Length')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Section 9: Export Cleaned Dataset

Save the cleaned and NLP-preprocessed DataFrame to a CSV file for downstream use.

In [ ]:
output_path = '../data/cleaned_customer_communication_dataset.csv'
df.to_csv(output_path, index=False)

print('=== Export Complete ===')
print(f'File saved to: {output_path}')
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nNew NLP columns added: cleaned_message, cleaned_word_count, cleaned_message_length')
print('\nDataset is ready for NLP model creation.')